# Virtual Embryo Challenge: от нуля до валидного файла сабмита (на русском)

Неофициальный стартовый ноутбук для [Virtual Embryo Challenge](https://virtualembryo.ai/challenge) (NeurIPS 2026, Qiu Lab, Стэнфорд).
Что он делает, по шагам:

1. ставит официальный локальный скорер `veckit`;
2. скачивает **живой контракт досок** с сайта (`index.json` + панели генов), так что ноутбук не устаревает при смене фазы;
3. загружает данные: по умолчанию крошечные публичные образцы из репозитория `veckit` (150 клеток), либо ваши настоящие файлы после регистрации;
4. строит опубликованные организаторами простые бейзлайны (`copy_last`, `pseudobulk_shift`, `wt_identity`, `shift_transfer`, `gene_ko`) для T1, T2 и T3;
5. приводит предсказание к формату конкретной доски (гены и их порядок, число клеток, `obsm["spatial_3D"]`) и проверяет его;
6. считает метрики локально через `veckit` на псевдо-разбиении (предсказать уже известную стадию по предыдущей).

**Честно о границах.** На образцах по 150 клеток метрики ничего не говорят о качестве модели, они только показывают, что конвейер работает.
Локальный счёт `veckit` не предсказывает ваш результат на доске: псевдо-цель взята из обучающих данных.
Этот ноутбук не содержит и не распространяет данные конкурса: образцы скачиваются из официального репозитория `veckit`, настоящие стадии вы скачиваете сами после входа на сайт.

Подготовлено с помощью ИИ-ассистента (Claude) и проверено полным запуском; все цифры ниже — реальный вывод ячеек.

## 1. Установка

In [1]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "veckit", "anndata", "scipy", "scikit-learn"], check=True)
import veckit, anndata, numpy, scipy
from importlib.metadata import version
print("veckit", version("veckit"), "| anndata", version("anndata"), "| numpy", numpy.__version__)

veckit 0.1.2 | anndata 0.13.4 | numpy 2.5.4


## 2. Настройки

`DATA_MODE = "sample"` — образцы из `veckit` (работает сразу, без регистрации).
`DATA_MODE = "real"` — ваши файлы, скачанные со страницы [Data](https://virtualembryo.ai/challenge/data) после входа (Google или GitHub).
Пропишите пути к ним в словаре `REAL` ниже. Имена файлов на сайте: `E8.5_RNA.h5ad`, `E9.5_RNA.h5ad` (T1), `E8.25_late.h5ad` и др. (T2), нокауты и дикий тип (T3).

In [2]:
from pathlib import Path

DATA_MODE = "sample"          # "sample" или "real"
WORK = Path("vec_work"); WORK.mkdir(exist_ok=True)
SEED = 0

# Пути для режима "real" (заполните своими файлами)
REAL = {
    "T1_prev":  "E8.5_RNA.h5ad",     # предпоследняя обучающая стадия T1
    "T1_last":  "E9.5_RNA.h5ad",     # последняя обучающая стадия T1
    "T2h_prev": "heart_E8.75.h5ad",  # heart: стадия перед последней
    "T2h_last": "heart_E9.5.h5ad",   # heart: последняя обучающая стадия
    "T3_ko_train": "mab21l2_ko_E9.5.h5ad",
    "T3_wt_train": "wt_E9.5.h5ad",
    "T3_wt_target": "wt_E8.75.h5ad", # дикий тип на стадии целевого нокаута
}

## 3. Живой контракт досок с сайта

In [3]:
import json, urllib.request

SITE = "https://virtualembryo.ai/challenge/panels/"
def fetch(url, dest):
    dest = WORK / dest
    if not dest.exists():
        # сайт отвечает 403 на пустой User-Agent, поэтому указываем его явно
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (vec-ru-starter)"})
        with urllib.request.urlopen(req, timeout=120) as r:
            dest.write_bytes(r.read())
    return dest

index = json.loads(fetch(SITE + "index.json", "index.json").read_text(encoding="utf-8"))
panels = {}
for key, b in index.items():
    genes = fetch(SITE + b["genes_file"], b["genes_file"]).read_text(encoding="utf-8").split()
    assert len(genes) == b["n_genes"], (key, len(genes), b["n_genes"])
    panels[key] = genes
    print(f'{key:24s} генов {b["n_genes"]:>6}  клеток {b["min_cells"]:>5}-{b["max_cells"]:<6} '
          f'координаты: {"да" if b["needs_coords"] else "нет"}  floor: {b["floor_model"]}')

T1:val                   генов  32285  клеток  1000-5118   координаты: нет  floor: copy_last


T2:embryo:val_interp     генов    498  клеток   583-5000   координаты: да  floor: copy_last


T2:heart:val_extrap      генов    500  клеток  1000-25179  координаты: да  floor: copy_last


T2:heart:val_interp      генов    500  клеток  1000-17616  координаты: да  floor: copy_last


T3:gata4                 генов    500  клеток  1000-7449   координаты: да  floor: wt_identity


Главное, что здесь видно и на чём чаще всего ошибаются:

* у каждой доски **своя** панель генов: `var_names` сравнивается поэлементно, в том же порядке. Брать панель нужно отсюда, а не из файла данных;
* у досок embryo (T2) **498** генов, а не 500: `Casp4` и `Pnliprp1` не измерены на E6.75 и E7.25 и исключены;
* T2 и T3 требуют `obsm["spatial_3D"]` формы `(n, 3)`; без координат файл отклоняется до подсчёта;
* число клеток — только в пределах диапазона доски. Это размер выборки, а не прогноз размера эмбриона: метрики сами берут подвыборку около 2 000 клеток, так что «пара тысяч» достаточно на любой доске.

С 20.10.2026 начинается финальная фаза P3: появятся тестовые доски, а этот `index.json` обновится. Ноутбук читает его при каждом запуске.

## 4. Данные

In [4]:
import anndata as ad

RAW = "https://raw.githubusercontent.com/aristoteleo/veckit/main/data/"
SAMPLE = {
    "T1_prev": "sample_8.5.h5ad", "T1_last": "sample_9.5.h5ad",
    "T2h_prev": "sample_heart_9.25.h5ad", "T2h_last": "sample_heart_9.5.h5ad",
    "T3_ko_train": "sample_mab21l2_ko.h5ad", "T3_wt_train": "sample_wt.h5ad",
    "T3_wt_target": "sample_wt.h5ad",   # в образцах нет WT E8.75, берём WT E9.5 как заглушку формата
}

def load(name):
    if DATA_MODE == "real":
        return ad.read_h5ad(REAL[name])
    return ad.read_h5ad(fetch(RAW + SAMPLE[name], SAMPLE[name]))

def src(name):
    # путь к файлу (для veckit.score)
    return REAL[name] if DATA_MODE == "real" else str(fetch(RAW + SAMPLE[name], SAMPLE[name]))

data = {k: load(k) for k in SAMPLE}
for k, a in data.items():
    print(f"{k:13s} {a.n_obs:>6} клеток x {a.n_vars:>6} генов | obs: {list(a.obs.columns)[:4]} | obsm: {list(a.obsm.keys())[:4]}")

T1_prev          150 клеток x  32285 генов | obs: ['celltype'] | obsm: ['X_umap.harmony.rna']
T1_last          150 клеток x  32285 генов | obs: ['celltype'] | obsm: ['X_umap.harmony.rna']
T2h_prev         150 клеток x    500 генов | obs: ['slide', 'section', 'n_counts', 'n_genes'] | obsm: ['X_pca', 'X_pca_harmony', 'X_umap', 'spatial']
T2h_last         150 клеток x    500 генов | obs: ['celltype', 'cm_celltype'] | obsm: ['spatial_2D', 'spatial_3D']
T3_ko_train      150 клеток x    500 генов | obs: ['slide', 'section', 'n_counts', 'n_genes'] | obsm: ['X_pca', 'X_pca_harmony', 'X_umap', 'spatial']
T3_wt_train      150 клеток x    500 генов | obs: ['celltype', 'cm_celltype'] | obsm: ['spatial_2D', 'spatial_3D']
T3_wt_target     150 клеток x    500 генов | obs: ['celltype', 'cm_celltype'] | obsm: ['spatial_2D', 'spatial_3D']


In [5]:
# Что внутри: типы клеток и их доли на двух стадиях T1
import pandas as pd
prev, last = data["T1_prev"], data["T1_last"]
comp = pd.concat([prev.obs["celltype"].value_counts(normalize=True).rename("prev"),
                  last.obs["celltype"].value_counts(normalize=True).rename("last")], axis=1).fillna(0)
shared = set(prev.obs["celltype"]) & set(last.obs["celltype"])
print("общих типов клеток:", len(shared), "из", comp.shape[0])
comp.sort_values("last", ascending=False).head(10).round(3)

общих типов клеток: 9 из 27


,prev,last
celltype,,
Endocardium,0.000,0.113
aSHF,0.033,0.093
Hepatocyte,0.000,0.093
SV-CM,0.020,0.087
Proepicardium,0.000,0.087
Foregut,0.173,0.080
V-CM,0.000,0.073
pSHF,0.140,0.053
OFT/RV-CM,0.060,0.053


Важная деталь из описания данных: словарь типов клеток **не согласован между стадиями** (например, `Endocardium` на E9.5 и `Endocardium-1/-2` позже).
Тип, «пропавший» на стадии, может просто называться иначе. Для бейзлайнов ниже это значит: типы без пары остаются как есть.

## 5. Бейзлайны организаторов (минимальные реализации)

In [6]:
import numpy as np
import scipy.sparse as sp

def dense(X):
    return np.asarray(X.toarray() if sp.issparse(X) else X, dtype=np.float32)

def copy_last(last):
    # T1/T2 floor: последняя наблюдаемая стадия как есть (для T2 вместе с координатами)
    return last.copy()

def pseudobulk_shift(prev, last):
    # каждой клетке прибавить сдвиг среднего её типа между двумя последними стадиями, обрезать в 0
    genes = last.var_names.intersection(prev.var_names, sort=False)
    Xl = dense(last[:, genes].X); Xp = dense(prev[:, genes].X)
    out = Xl.copy()
    for ct in pd.unique(last.obs["celltype"]):
        ml = (last.obs["celltype"] == ct).to_numpy()
        mp = (prev.obs["celltype"] == ct).to_numpy()
        if mp.sum() == 0:
            continue                      # типа нет на ранней стадии: оставить как есть
        delta = Xl[ml].mean(0) - Xp[mp].mean(0)
        out[ml] = np.clip(Xl[ml] + delta, 0, None)
    res = ad.AnnData(X=out, obs=last.obs[["celltype"]].copy(), var=pd.DataFrame(index=genes))
    for k in ("spatial_3D",):
        if k in last.obsm:
            res.obsm[k] = np.asarray(last.obsm[k])
    return res

def wt_identity(wt_target):
    # T3 floor: дикий тип целевой стадии, «нокаут ничего не меняет»
    return wt_target.copy()

def shift_transfer(ko_train, wt_train, wt_target):
    # T3: перенести псевдобалк-сдвиг обучающего нокаута на дикий тип целевой стадии
    genes = wt_target.var_names
    delta = dense(ko_train[:, genes].X).mean(0) - dense(wt_train[:, genes].X).mean(0)
    res = wt_target.copy()
    res.X = np.clip(dense(wt_target.X) + delta, 0, None)
    return res

def gene_ko(pred, gene):
    # shift_transfer + обнулить сам выключенный ген
    res = pred.copy()
    X = dense(res.X)
    if gene in res.var_names:
        X[:, res.var_names.get_loc(gene)] = 0.0
    res.X = X
    return res
print("ok")

ok


## 6. Привести к формату доски и проверить

`conform` делает ровно то, что требует контракт: гены доски в её порядке (недостающие заполняются нулями, с предупреждением),
неотрицательная конечная `float32`-матрица, координаты для T2/T3, число клеток в диапазоне.
Если клеток больше максимума, берётся случайная подвыборка. Если меньше минимума (бывает только на образцах по 150 клеток), клетки дублируются: это **только для демонстрации формата**, в настоящем сабмите так делать не нужно, у реальных стадий клеток хватает.

In [7]:
def conform(pred, board, n_cells=None, seed=SEED):
    b, genes = index[board], panels[board]
    rng = np.random.default_rng(seed)
    missing = [g for g in genes if g not in pred.var_names]
    if missing:
        print(f"  внимание: {len(missing)} генов панели нет в предсказании, заполняю нулями: {missing[:5]}")
    X = dense(pred.X)
    cols = {g: i for i, g in enumerate(pred.var_names)}
    Y = np.zeros((pred.n_obs, len(genes)), dtype=np.float32)
    for j, g in enumerate(genes):
        if g in cols:
            Y[:, j] = X[:, cols[g]]
    Y = np.clip(np.nan_to_num(Y, nan=0.0, posinf=0.0, neginf=0.0), 0, None)
    n = n_cells or min(max(pred.n_obs, b["min_cells"]), b["max_cells"])
    idx = rng.choice(pred.n_obs, size=n, replace=n > pred.n_obs)
    if n > pred.n_obs:
        print(f"  внимание: в данных {pred.n_obs} клеток < {n}, дублирую (только демо формата)")
    out = ad.AnnData(X=sp.csr_matrix(Y[idx]), var=pd.DataFrame(index=pd.Index(genes)))
    out.obs_names = [f"pred_{i}" for i in range(n)]
    if b["needs_coords"]:
        out.obsm["spatial_3D"] = np.asarray(pred.obsm["spatial_3D"], dtype=np.float32)[idx, :3]
    return out

def check(a, board):
    b, genes = index[board], panels[board]
    errs = []
    if list(a.var_names) != genes: errs.append("var_names не совпадают с панелью доски (состав или порядок)")
    if not (b["min_cells"] <= a.n_obs <= b["max_cells"]): errs.append(f"клеток {a.n_obs}, нужно {b['min_cells']}-{b['max_cells']}")
    X = a.X.data if sp.issparse(a.X) else np.asarray(a.X)
    if not np.isfinite(X).all(): errs.append("в X есть NaN/inf")
    if (X < 0).any(): errs.append("в X есть отрицательные значения")
    for k in b["obsm_required"]:
        if k not in a.obsm or np.asarray(a.obsm[k]).shape != (a.n_obs, 3) or not np.isfinite(a.obsm[k]).all():
            errs.append(f'obsm["{k}"] отсутствует или не (n, 3) / не конечный')
    print(f"{board}: " + ("OK" if not errs else "ОШИБКИ: " + "; ".join(errs)))
    return not errs

## 7. Task 1: E10.5 по E8.5 и E9.5

Сначала файл для доски `T1:val`, затем локальный счёт на псевдо-разбиении: «предсказать последнюю известную стадию по предпоследней».

In [8]:
files = {}
p = conform(pseudobulk_shift(data["T1_prev"], data["T1_last"]), "T1:val")
files["T1:val"] = WORK / "T1_val_pseudobulk_shift.h5ad"; p.write_h5ad(files["T1:val"])
check(p, "T1:val")

  внимание: в данных 150 клеток < 1000, дублирую (только демо формата)


T1:val: OK


True

In [9]:
from veckit import score
import warnings; warnings.filterwarnings("ignore")

# псевдо-разбиение: предсказание = предпоследняя стадия (copy_last), цель = последняя, reference = предпоследняя
r = score(task="T1", input=src("T1_prev"), target=src("T1_last"), reference=src("T1_prev"))
pd.Series({k: v for k, v in r["metrics"].items() if not k.startswith("_")}).round(4)

de_score              0.0000
de_direction          0.0000
energy_distance       0.3650
mmd_u                 0.0772
variogram             0.0019
pb_rel_err            0.1211
library_size_ratio    1.0000
variance_ratio        0.9740
composition_JSD       0.0266
pseudobulk_pearson    0.9915
dtype: float64

Как читать: `de_score` и `de_direction` — «угадали ли вы, какие гены и куда сдвинулись». У `copy_last` они равны 0 по построению: он ничего не меняет.
`mmd_u`, `energy_distance`, `variogram` — расстояния между распределениями (меньше — лучше). Веса T1 на сайте: DE-гены 25 %, направление 25 %, распределение состояний 30 %, ко-вариация генов 20 %.
Ловушка, о которой предупреждают организаторы: одна «средняя клетка», размноженная до нужного числа, хорошо проходит метрики по средним и почти обнуляет распределительные.

## 8. Task 2 (heart): координаты обязательны

In [10]:
p = conform(copy_last(data["T2h_last"]), "T2:heart:val_extrap")
files["T2:heart:val_extrap"] = WORK / "T2_heart_val_extrap_copy_last.h5ad"; p.write_h5ad(files["T2:heart:val_extrap"])
check(p, "T2:heart:val_extrap")

r = score(task="T2", setting="heart", input=src("T2h_prev"), target=src("T2h_last"), reference=src("T2h_prev"))
pd.Series({k: v for k, v in r["metrics"].items() if not k.startswith("_")}).round(4)

  внимание: в данных 150 клеток < 1000, дублирую (только демо формата)
T2:heart:val_extrap: OK


de_score               0.0000
de_direction           0.0000
energy_distance       11.7040
mmd_u                  0.2904
variogram              0.0981
d2_shape               0.0269
sliced_wasserstein     0.0609
occupancy_dice         0.5000
size_fidelity          0.9269
scale_ratio            0.9284
scale_log_ratio       -0.0742
count_log_ratio        0.0000
neighborhood_mmd       0.5840
pb_rel_err             0.7653
library_size_ratio     0.0170
variance_ratio         0.2010
composition_JSD        0.0282
pseudobulk_pearson     0.7907
morans_I_agreement     0.6347
dtype: float64

In [11]:
# Проверка масштаба: две стадии должны быть в одной нормировке (log1p). Иначе метрики «сдвига» бессмысленны.
for k in ("T2h_prev", "T2h_last"):
    X = dense(data[k].X)
    print(f"{k:9s} средняя сумма по клетке {X.sum(1).mean():8.1f}   максимум {X.max():6.2f}")

T2h_prev  средняя сумма по клетке     35.8   максимум  63.00
T2h_last  средняя сумма по клетке    194.5   максимум   8.43


Если суммы и максимумы двух стадий различаются в разы (в публичных образцах `veckit` так и есть: у `sample_heart_9.25` максимум около 63, у остальных файлов около 8, а `library_size_ratio` в счёте выше около 0.02), то файлы лежат в разной нормировке. На образцах это влияет только на демо-счёт; на настоящих файлах проверьте то же самое до обучения.

Координаты в T2/T3 локальны для каждого эмбриона и не согласованы между стадиями; метрики формы инвариантны к сдвигу и повороту.
Известное слепое пятно, описанное в FAQ: зеркальный эмбрион пока получает тот же счёт, что и правильный.

## 9. Task 3: нокаут Gata4 по нокауту Mab21l2

In [12]:
pred = gene_ko(shift_transfer(data["T3_ko_train"], data["T3_wt_train"], data["T3_wt_target"]), "Gata4")
p = conform(pred, "T3:gata4")
files["T3:gata4"] = WORK / "T3_gata4_gene_ko.h5ad"; p.write_h5ad(files["T3:gata4"])
check(p, "T3:gata4")

# псевдо-проверка счёта: wt_identity против известного нокаута Mab21l2 (reference = дикий тип)
r = score(task="T3", input=src("T3_wt_train"), target=src("T3_ko_train"), wt=src("T3_wt_train"))
pd.Series({k: v for k, v in r["metrics"].items() if not k.startswith("_")}).round(4)

  внимание: в данных 150 клеток < 1000, дублирую (только демо формата)
T3:gata4: OK


de_score              0.0000
de_direction          0.0000
severity_ratio        0.0000
severity_slope       -6.9078
energy_distance       1.0714
mmd_u                 0.0462
variogram             0.0326
pb_rel_err            0.3727
library_size_ratio    1.0000
variance_ratio        0.9210
composition_JSD       0.0202
pseudobulk_pearson    0.9033
d2_shape              0.0192
sliced_wasserstein    0.0618
occupancy_dice        0.5263
scale_log_ratio       0.1045
count_log_ratio       0.0000
neighborhood_mmd      0.1339
dtype: float64

В T3 счёт считается по **разнице** «дикий тип → мутант», а не по абсолютной экспрессии: вернуть дикий тип без изменений уже даёт корреляцию 0.956 по абсолютным значениям, поэтому это и есть нулевой уровень (floor).
В режиме `sample` дикий тип E8.75 недоступен, и вместо него взят E9.5: файл `T3:gata4` здесь демонстрирует только формат.

## 10. Итог и загрузка

In [13]:
for board, f in files.items():
    a = ad.read_h5ad(f)
    ok = check(a, board)
    print(f"   {f}  {f.stat().st_size/1e6:.1f} МБ  {a.n_obs} клеток")

T1:val: OK
   vec_work\T1_val_pseudobulk_shift.h5ad  48.4 МБ  1000 клеток
T2:heart:val_extrap: OK
   vec_work\T2_heart_val_extrap_copy_last.h5ad  0.4 МБ  1000 клеток
T3:gata4: OK
   vec_work\T3_gata4_gene_ko.h5ad  2.6 МБ  1000 клеток


Дальше:

1. переключите `DATA_MODE = "real"`, укажите пути к своим файлам и перезапустите всё;
2. загрузите `.h5ad` на странице [Submit](https://virtualembryo.ai/challenge) выбранной доски. Проверка формата на сайте бесплатна и не тратит попытку;
3. лимиты: P1 — 20 оценённых сабмитов в день на задачу, P2 — 8, финальная P3 (с 20.10.2026) — 2 официальных на доску на всю фазу; файл не больше 1200 МБ;
4. если используете внешние данные (поощряется в T1) — раскройте каждый источник в описании метода. Данные с отложенных стадий и генотипов запрещены любым путём, включая «соседние» стадии; окна перечислены в [правилах](https://virtualembryo.ai/challenge/rules), раздел 10.

Вопросы — в Slack (#virtual-embryo-challenge), Discord или почтой организаторам; ссылки на [странице FAQ](https://virtualembryo.ai/challenge/faq).